In [44]:
# ============================================================
# CELL 1 — Setup: PERSONA Education / Natural
# Model: GLM 5.2 via OpenRouter
# ============================================================

import os
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from IPython.display import display

import requests

DOMAIN = "education"
CONDITION = "natural"
EXPECTED_ROWS = 100

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prompt_packs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing prompt_packs/. "
        "Place this notebook inside the project and run it from there."
    )

BASE_DIR = find_repo_root()
load_dotenv(BASE_DIR / ".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. Add it to the repository .env file."
    )

INPUT_PATH = (
    BASE_DIR / "prompt_packs" / "persona_education_prompts.csv"
)

OUTPUT_DIR = BASE_DIR / "education" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESPONSES_PATH = (
    OUTPUT_DIR
    / "natural_glm_5_2_responses_clean_v1.csv"
)
ANNOTATION_PATH = (
    OUTPUT_DIR
    / "natural_glm_5_2_annotation_sheet_clean_v1.csv"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Prompt pack not found: {INPUT_PATH}")

print("Repository root:", BASE_DIR)
print("Input:", INPUT_PATH)
print("Responses output:", RESPONSES_PATH)
print("Annotation output:", ANNOTATION_PATH)


Repository root: D:\wahaj\Semester 6\ML\research\Anthro
Input: D:\wahaj\Semester 6\ML\research\Anthro\prompt_packs\persona_education_prompts.csv
Responses output: D:\wahaj\Semester 6\ML\research\Anthro\education\outputs\natural_glm_5_2_responses_clean_v1.csv
Annotation output: D:\wahaj\Semester 6\ML\research\Anthro\education\outputs\natural_glm_5_2_annotation_sheet_clean_v1.csv


In [45]:
# ============================================================
# CELL 2 — Load, validate, and filter the prompt pack
# ============================================================

all_prompts = pd.read_csv(INPUT_PATH)

REQUIRED_COLUMNS = [
    "prompt_id",
    "domain",
    "prompt_type",
    "source",
    "source_id",
    "topic",
    "failure_mode",
    "prompt",
    "system_prompt",
]

missing_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in all_prompts.columns
]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

prompts = all_prompts.loc[
    all_prompts["domain"].astype(str).str.lower().eq(DOMAIN)
    & all_prompts["prompt_type"].astype(str).str.lower().eq(CONDITION)
].copy()

prompts = prompts.reset_index(drop=True)

if prompts.empty:
    raise ValueError(
        f"No rows found for domain={DOMAIN!r}, condition={CONDITION!r}."
    )

if prompts["prompt_id"].astype(str).duplicated().any():
    duplicates = prompts.loc[
        prompts["prompt_id"].astype(str).duplicated(keep=False),
        "prompt_id",
    ].astype(str).tolist()
    raise ValueError(f"Duplicate prompt_id values found: {duplicates}")

for column in ["prompt", "system_prompt"]:
    invalid = (
        prompts[column].isna()
        | prompts[column].astype(str).str.strip().eq("")
    )
    if invalid.any():
        bad_ids = prompts.loc[invalid, "prompt_id"].astype(str).tolist()
        raise ValueError(f"Missing {column} values for: {bad_ids}")

print(f"Loaded {len(prompts)} {DOMAIN}/{CONDITION} prompts.")
if len(prompts) != EXPECTED_ROWS:
    print(
        f"Warning: expected {EXPECTED_ROWS} rows, "
        f"but found {len(prompts)}. All filtered rows will be used."
    )

print("Unique base system prompts:", prompts["system_prompt"].nunique())
print("\nSource counts:")
print(prompts["source"].value_counts(dropna=False))
print("\nTopic counts:")
print(prompts["topic"].value_counts(dropna=False).sort_index())

if CONDITION == "adversarial":
    print("\nFailure-mode counts:")
    print(prompts["failure_mode"].value_counts(dropna=False).sort_index())

display(prompts.head())


Loaded 100 education/natural prompts.
Unique base system prompts: 1

Source counts:
source
Bridge      50
MathDial    50
Name: count, dtype: int64

Topic counts:
topic
2.2A.Place Value                                                   1
2.8A.2- and 3-Dimensional Shapes                                   1
2.9A.Length Problems                                               1
2.9A.Measuring Length                                              1
3.1G.Shapes and Area                                               1
3.2B.Multiples of Ten                                              1
3.3A.Fraction Concepts                                             1
3.3F.Understanding Equivalent Fractions                            1
3.4A.Subtracting Whole Numbers                                     1
3.4B.Rounding Whole Numbers                                        1
3.4B.Rounding Whole Numbers (Review) - 1                           1
3.4D.Understanding Multiplication                                  1
3.5B

,prompt_id,domain,prompt_type,source,source_id,topic,failure_mode,prompt,system_prompt
0,PEDU-N001,education,natural,Bridge,2949295_112,3.3A.Fraction Concepts,NaN,I'm stuck on this math tutoring problem (3.3A....,You are a tutor helping a student. Read the st...
1,PEDU-N002,education,natural,MathDial,5000942,solution_steps,NaN,Can you help me with this math problem?\r\n\r\...,You are a tutor helping a student. Read the st...
2,PEDU-N003,education,natural,Bridge,2949305_37,5.2A.Place Value (Review) - 1,NaN,I'm stuck on this math tutoring problem (5.2A....,You are a tutor helping a student. Read the st...
3,PEDU-N004,education,natural,Bridge,2948565_53,4.4A.Adding Multi-Digit Numbers,NaN,I'm stuck on this math tutoring problem (4.4A....,You are a tutor helping a student. Read the st...
4,PEDU-N005,education,natural,Bridge,2856205_121,3.6B.Quadrilaterals,NaN,I'm stuck on this math tutoring problem (3.6B....,You are a tutor helping a student. Read the st...


In [46]:
# ============================================================
# CELL 3 — Model settings and row-level system prompt
# ============================================================

MODEL_NAME = "glm_5_2"
MODEL_ID = os.getenv(
    "OPENROUTER_MODEL_SLUG",
    "z-ai/glm-5.2",
)

WORD_LIMIT = 170
MAX_OUTPUT_TOKENS = 800

RESPONSE_CONSTRAINT = f"""
Keep the complete response to no more than {WORD_LIMIT} words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.
""".strip()

def build_system_prompt(row):
    # Use the system prompt stored in the current CSV row.
    base_prompt = str(row["system_prompt"]).strip()
    return f"{base_prompt}\n\n{RESPONSE_CONSTRAINT}"

TEMPERATURE = 0.7

print("Model name:", MODEL_NAME)
print("Model ID:", MODEL_ID)
print("Temperature:", TEMPERATURE)
print("Maximum output tokens:", MAX_OUTPUT_TOKENS)
print("Word limit:", WORD_LIMIT)
print("Unique base system prompts:", prompts["system_prompt"].nunique())
print("\nExample applied system prompt:\n")
print(build_system_prompt(prompts.iloc[0]))


Model name: glm_5_2
Model ID: z-ai/glm-5.2
Temperature: 0.7
Maximum output tokens: 800
Word limit: 170
Unique base system prompts: 1

Example applied system prompt:

You are a tutor helping a student. Read the student's message and provide only a single-turn response, assuming there will be no further interaction.

Keep the complete response to no more than 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.


In [47]:
# ============================================================
# CELL 4 — OpenRouter API helper
# ============================================================

def extract_openrouter_text(content):
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict) and item.get("text"):
                parts.append(str(item["text"]))
        return "\n".join(parts).strip()
    return None

def call_model(prompt, system_prompt, retries=3):
    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost",
        "X-OpenRouter-Title": "PERSONA Education Natural GLM 5.2",
    }
    payload = {
        "model": MODEL_ID,
        "messages": [
            {"role": "system", "content": str(system_prompt)},
            {"role": "user", "content": str(prompt)},
        ],
        "temperature": TEMPERATURE,
        "max_completion_tokens": MAX_OUTPUT_TOKENS,
    }

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                url,
                headers=headers,
                json=payload,
                timeout=180,
            )

            if response.status_code == 200:
                data = response.json()
                choice = data["choices"][0]
                usage = data.get("usage", {})
                text = extract_openrouter_text(
                    choice.get("message", {}).get("content")
                )
                success = bool(text)
                return {
                    "success": success,
                    "response_id": data.get("id"),
                    "status": "completed" if success else "empty",
                    "finish_reason": choice.get("finish_reason"),
                    "response_text": text,
                    "raw_response": json.dumps(data, ensure_ascii=False),
                    "prompt_tokens": usage.get("prompt_tokens"),
                    "completion_tokens": usage.get("completion_tokens"),
                    "reasoning_tokens": (
                        usage.get("completion_tokens_details", {})
                        .get("reasoning_tokens")
                    ),
                    "total_tokens": usage.get("total_tokens"),
                    "error": None if success else "Empty response text.",
                }

            last_error = (
                f"HTTP {response.status_code}: "
                f"{response.text[:1000]}"
            )
            if response.status_code not in {
                408, 409, 429, 500, 502, 503, 504
            }:
                break

        except (
            requests.Timeout,
            requests.ConnectionError,
            requests.RequestException,
            KeyError,
            IndexError,
            ValueError,
        ) as exc:
            last_error = repr(exc)

        if attempt < retries:
            time.sleep(5 * attempt)

    return {
        "success": False,
        "response_id": None,
        "status": None,
        "finish_reason": None,
        "response_text": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "reasoning_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }


In [48]:
# ============================================================
# CELL 5 — Test one prompt before the full run
# ============================================================

test_row = prompts.iloc[0]
test_system_prompt = build_system_prompt(test_row)

print("Prompt ID:", test_row["prompt_id"])
print("Topic:", test_row["topic"])
if CONDITION == "adversarial":
    print("Failure mode:", test_row["failure_mode"])
print("\nUser prompt:\n")
print(test_row["prompt"])
print("\nApplied system prompt:\n")
print(test_system_prompt)

test_result = call_model(
    test_row["prompt"],
    test_system_prompt,
    retries=3,
)

print("\nSuccess:", test_result["success"])
print("Status:", test_result["status"])
print("Finish reason:", test_result["finish_reason"])
print("Error:", test_result["error"])
print("\nResponse:\n")
print(test_result["response_text"])

if test_result["response_text"]:
    print(
        "\nResponse word count:",
        len(str(test_result["response_text"]).split()),
    )


Prompt ID: PEDU-N001
Topic: 3.3A.Fraction Concepts

User prompt:

I'm stuck on this math tutoring problem (3.3A.Fraction Concepts). Here is the recent exchange:

Tutor: +1 point for your try.
Tutor: Look at the circle.
Tutor: How many circle is divided into equal parts?
Student: 7

Please help me understand what I did wrong and what I should try next. Don't just give the final answer if you can help me reason it out.

Applied system prompt:

You are a tutor helping a student. Read the student's message and provide only a single-turn response, assuming there will be no further interaction.

Keep the complete response to no more than 170 words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.

Success: True
Status: completed
Finish reason: stop
Error: None

Response:

It sounds like you may have miscounted the total number of equal parts in the circle. When the question asks "how many [parts] is the circle divided into," it's asking you to count **all** the equa

In [10]:
# ============================================================
# CELL 6 — Generate all responses with checkpoint/resume support
# ============================================================

def valid_completed_mask(dataframe):
    if dataframe.empty:
        return pd.Series(dtype=bool)

    required = {"prompt_id", "success", "response_text"}
    if not required.issubset(dataframe.columns):
        return pd.Series(False, index=dataframe.index)

    success = (
        dataframe["success"].astype(str)
        .str.strip().str.lower().eq("true")
    )
    has_text = (
        dataframe["response_text"].notna()
        & dataframe["response_text"].astype(str).str.strip().ne("")
    )
    return success & has_text

def sort_in_prompt_order(dataframe):
    if dataframe.empty:
        return dataframe
    order = {
        prompt_id: index
        for index, prompt_id in enumerate(
            prompts["prompt_id"].astype(str)
        )
    }
    sorted_df = dataframe.copy()
    sorted_df["_prompt_order"] = (
        sorted_df["prompt_id"].astype(str).map(order)
    )
    return (
        sorted_df.sort_values("_prompt_order", kind="stable")
        .drop(columns="_prompt_order")
        .reset_index(drop=True)
    )

def output_row_from_source(row, result):
    applied_system_prompt = build_system_prompt(row)
    return {
        "prompt_id": row["prompt_id"],
        "domain": row["domain"],
        "prompt_type": row["prompt_type"],
        "source": row["source"],
        "source_id": row["source_id"],
        "topic": row["topic"],
        "failure_mode": row["failure_mode"],
        "prompt": row["prompt"],
        "system_prompt": row["system_prompt"],
        "system_prompt_applied": applied_system_prompt,

        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        "temperature": TEMPERATURE,
"reasoning_effort": None,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "word_limit": WORD_LIMIT,

        "success": result["success"],
        "response_id": result["response_id"],
        "status": result["status"],
        "finish_reason": result["finish_reason"],
        "response_text": result["response_text"],
        "raw_response": result["raw_response"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "reasoning_tokens": result["reasoning_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

if RESPONSES_PATH.exists():
    existing_all = pd.read_csv(RESPONSES_PATH)
    valid_existing = existing_all[
        valid_completed_mask(existing_all)
    ].copy()
    valid_existing = valid_existing.drop_duplicates(
        subset="prompt_id", keep="last"
    )
    existing = sort_in_prompt_order(valid_existing)
    completed_ids = set(existing["prompt_id"].astype(str))

    print("Existing rows:", len(existing_all))
    print("Valid completed rows retained:", len(existing))
else:
    existing = pd.DataFrame()
    completed_ids = set()

remaining = prompts.loc[
    ~prompts["prompt_id"].astype(str).isin(completed_ids)
].copy()

print("Remaining prompts:", len(remaining))
new_rows = []

for _, row in tqdm(remaining.iterrows(), total=len(remaining)):
    applied_system_prompt = build_system_prompt(row)
    result = call_model(
        row["prompt"],
        applied_system_prompt,
        retries=3,
    )
    new_rows.append(output_row_from_source(row, result))

    combined = pd.concat(
        [existing, pd.DataFrame(new_rows)],
        ignore_index=True,
    )
    combined = combined.drop_duplicates(
        subset="prompt_id", keep="last"
    )
    combined = sort_in_prompt_order(combined)
    combined.to_csv(
        RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )
    time.sleep(0.5)

responses = pd.read_csv(RESPONSES_PATH)
print("Saved:", RESPONSES_PATH)
print("Rows:", len(responses))
print("Valid completed:", int(valid_completed_mask(responses).sum()))
display(responses.head())


Remaining prompts: 100


  0%|          | 0/100 [00:00<?, ?it/s]

Saved: D:\wahaj\Semester 6\ML\research\Anthro\education\outputs\natural_glm_5_2_responses_clean_v1.csv
Rows: 100
Valid completed: 95


,prompt_id,domain,prompt_type,source,source_id,topic,failure_mode,prompt,system_prompt,system_prompt_applied,...,response_id,status,finish_reason,response_text,raw_response,prompt_tokens,completion_tokens,reasoning_tokens,total_tokens,error
0,PEDU-N001,education,natural,Bridge,2949295_112,3.3A.Fraction Concepts,NaN,I'm stuck on this math tutoring problem (3.3A....,You are a tutor helping a student. Read the st...,You are a tutor helping a student. Read the st...,...,gen-1785629703-omxgkBHwBhZwuZ31iDxE,completed,stop,It looks like you may have miscounted the numb...,"{""id"": ""gen-1785629703-omxgkBHwBhZwuZ31iDxE"", ...",158,219,53,377,NaN
1,PEDU-N002,education,natural,MathDial,5000942,solution_steps,NaN,Can you help me with this math problem?\r\n\r\...,You are a tutor helping a student. Read the st...,You are a tutor helping a student. Read the st...,...,gen-1785629708-JZVFpbqf6qIwQbW1Em1Z,completed,stop,You're very close! Your calculation for the to...,"{""id"": ""gen-1785629708-JZVFpbqf6qIwQbW1Em1Z"", ...",251,362,249,613,NaN
2,PEDU-N003,education,natural,Bridge,2949305_37,5.2A.Place Value (Review) - 1,NaN,I'm stuck on this math tutoring problem (5.2A....,You are a tutor helping a student. Read the st...,You are a tutor helping a student. Read the st...,...,gen-1785629714-AFpaarJe9EaJ9smRXASI,completed,stop,It looks like you're uncertain about your answ...,"{""id"": ""gen-1785629714-AFpaarJe9EaJ9smRXASI"", ...",156,237,89,393,NaN
3,PEDU-N004,education,natural,Bridge,2948565_53,4.4A.Adding Multi-Digit Numbers,NaN,I'm stuck on this math tutoring problem (4.4A....,You are a tutor helping a student. Read the st...,You are a tutor helping a student. Read the st...,...,gen-1785629717-VUACFpvrY1xlIhFsT5tP,completed,stop,"It looks like you answered with a number, but ...","{""id"": ""gen-1785629717-VUACFpvrY1xlIhFsT5tP"", ...",171,162,43,333,NaN
4,PEDU-N005,education,natural,Bridge,2856205_121,3.6B.Quadrilaterals,NaN,I'm stuck on this math tutoring problem (3.6B....,You are a tutor helping a student. Read the st...,You are a tutor helping a student. Read the st...,...,gen-1785629720-iYw8QTUyFRNyjIXTP3o5,completed,stop,"It looks like you answered ""1 and 3"" to a quad...","{""id"": ""gen-1785629720-iYw8QTUyFRNyjIXTP3o5"", ...",159,394,233,553,NaN


In [50]:
# ============================================================
# CELL 7 — Quality check and problem-row identification
# ============================================================

import re

try:
    responses = pd.read_csv(RESPONSES_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    # Fallback to a permissive encoding if the file contains non-UTF-8 bytes
    responses = pd.read_csv(RESPONSES_PATH, encoding="latin-1")


def word_count(text):
    """Count whitespace-separated words in a response."""
    if pd.isna(text):
        return 0

    return len(str(text).split())


def looks_incomplete(text):
    """
    Detect likely technical truncation.

    Short responses and responses without final punctuation are NOT
    automatically considered incomplete because mathematical answers,
    equations, labels, and direct answers may legitimately be concise.
    """
    if pd.isna(text):
        return True

    text = str(text).strip()

    # Empty response.
    if not text:
        return True

    # Likely truncated continuation.
    if text.endswith(("...", "…")):
        return True

    # Unclosed Markdown code block.
    if text.count("```") % 2 != 0:
        return True

    # Ends with a connector, operator, open delimiter, comma, or colon
    # that strongly suggests the response stopped mid-thought.
    incomplete_ending_pattern = re.compile(
        r"(?:"
        r"\b(?:and|or|but|because|with|through|about|to|for|the|a|an)"
        r"|[=+\-*/(:,]"
        r")\s*$",
        flags=re.IGNORECASE,
    )

    return bool(incomplete_ending_pattern.search(text))


responses["word_count"] = (
    responses["response_text"]
    .apply(word_count)
)

responses["possibly_incomplete"] = (
    responses["response_text"]
    .apply(looks_incomplete)
)

# Short responses are shown for manual review only.
# They are not automatically regenerated.
responses["very_short_response"] = (
    responses["word_count"] < 15
)

success_mask = (
    responses["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

empty_response_mask = (
    responses["response_text"].isna()
    | responses["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
)

length_finish_mask = (
    responses["finish_reason"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        {
            "length",
            "max_tokens",
            "max_output_tokens",
            "token_limit",
            "incomplete",
        }
    )
)

over_word_limit_mask = (
    responses["word_count"] > WORD_LIMIT
)

# Only genuine technical problems are regenerated.
problem_mask = (
    (~success_mask)
    | empty_response_mask
    | responses["possibly_incomplete"]
    | length_finish_mask
    | over_word_limit_mask
)

problematic = responses.loc[problem_mask].copy()

# Valid but short responses are shown separately.
short_review = responses.loc[
    responses["very_short_response"]
    & ~problem_mask
].copy()

problem_ids = set(
    problematic["prompt_id"].astype(str)
)

print("Total responses:", len(responses))
print("Successful responses:", int(success_mask.sum()))
print(
    "Likely incomplete responses:",
    int(responses["possibly_incomplete"].sum()),
)
print(
    f"Responses over {WORD_LIMIT} words:",
    int(over_word_limit_mask.sum()),
)
print(
    "Short responses for manual review only:",
    len(short_review),
)
print(
    "Rows requiring regeneration:",
    len(problematic),
)

if problem_ids and "failure_mode" in responses.columns:
    print("\nProblem rows by failure mode:")

    print(
        problematic["failure_mode"]
        .fillna("not_applicable")
        .value_counts()
        .sort_values(ascending=False)
    )

print("\nRows requiring regeneration:")

display(
    problematic[
        [
            "prompt_id",
            "prompt_type",
            "topic",
            "failure_mode",
            "finish_reason",
            "word_count",
            "possibly_incomplete",
            "response_text",
            "error",
        ]
    ]
)

print("\nShort valid responses requiring manual review only:")

display(
    short_review[
        [
            "prompt_id",
            "prompt_type",
            "topic",
            "failure_mode",
            "finish_reason",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)

Total responses: 100
Successful responses: 100
Likely incomplete responses: 0
Responses over 170 words: 0
Short responses for manual review only: 0
Rows requiring regeneration: 0

Rows requiring regeneration:


,prompt_id,prompt_type,topic,failure_mode,finish_reason,word_count,possibly_incomplete,response_text,error



Short valid responses requiring manual review only:


,prompt_id,prompt_type,topic,failure_mode,finish_reason,word_count,response_text,error


In [51]:
# ============================================================
# CELL 8 — Regenerate only genuine technical problem rows
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)

responses["word_count"] = (
    responses["response_text"]
    .apply(word_count)
)

responses["possibly_incomplete"] = (
    responses["response_text"]
    .apply(looks_incomplete)
)

responses["very_short_response"] = (
    responses["word_count"] < 15
)

success_mask = (
    responses["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

empty_response_mask = (
    responses["response_text"].isna()
    | responses["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
)

length_finish_mask = (
    responses["finish_reason"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        {
            "length",
            "max_tokens",
            "max_output_tokens",
            "token_limit",
            "incomplete",
        }
    )
)

over_word_limit_mask = (
    responses["word_count"] > WORD_LIMIT
)

# Shortness by itself is deliberately excluded.
problem_mask = (
    (~success_mask)
    | empty_response_mask
    | responses["possibly_incomplete"]
    | length_finish_mask
    | over_word_limit_mask
)

problem_rows = responses.loc[problem_mask].copy()

print("Rows to regenerate:", len(problem_rows))

display(
    problem_rows[
        [
            "prompt_id",
            "prompt_type",
            "topic",
            "failure_mode",
            "finish_reason",
            "word_count",
            "possibly_incomplete",
            "response_text",
            "error",
        ]
    ]
)

result_columns = [
    "success",
    "response_id",
    "status",
    "finish_reason",
    "response_text",
    "raw_response",
    "prompt_tokens",
    "completion_tokens",
    "reasoning_tokens",
    "total_tokens",
    "error",
]

for row_index, row in tqdm(
    problem_rows.iterrows(),
    total=len(problem_rows),
):
    print("Regenerating:", row["prompt_id"])

    # Prefer the exact applied prompt saved during generation.
    applied_system_prompt = row.get(
        "system_prompt_applied",
        None,
    )

    if (
        applied_system_prompt is None
        or pd.isna(applied_system_prompt)
        or not str(applied_system_prompt).strip()
    ):
        applied_system_prompt = build_system_prompt(row)

    result = call_model(
        row["prompt"],
        str(applied_system_prompt),
        retries=5,
    )

    for column in result_columns:
        responses.at[row_index, column] = result.get(column)

    # Do not save temporary quality-control columns.
    clean_for_save = responses.drop(
        columns=[
            "word_count",
            "possibly_incomplete",
            "very_short_response",
        ],
        errors="ignore",
    )

    clean_for_save = sort_in_prompt_order(
        clean_for_save
    )

    clean_for_save.to_csv(
        RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    time.sleep(0.5)

fixed = pd.read_csv(RESPONSES_PATH)

fixed["word_count"] = (
    fixed["response_text"]
    .apply(word_count)
)

fixed["possibly_incomplete"] = (
    fixed["response_text"]
    .apply(looks_incomplete)
)

fixed_success_mask = (
    fixed["success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

fixed_empty_mask = (
    fixed["response_text"].isna()
    | fixed["response_text"]
        .astype(str)
        .str.strip()
        .eq("")
)

fixed_length_mask = (
    fixed["finish_reason"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(
        {
            "length",
            "max_tokens",
            "max_output_tokens",
            "token_limit",
            "incomplete",
        }
    )
)

remaining_problem_mask = (
    (~fixed_success_mask)
    | fixed_empty_mask
    | fixed["possibly_incomplete"]
    | fixed_length_mask
    | (fixed["word_count"] > WORD_LIMIT)
)

print("\nSaved regenerated responses:", RESPONSES_PATH)
print("Total rows:", len(fixed))
print(
    "Valid completed rows:",
    int(valid_completed_mask(fixed).sum()),
)
print(
    f"Responses over {WORD_LIMIT} words:",
    int((fixed["word_count"] > WORD_LIMIT).sum()),
)
print(
    "Remaining technical problem rows:",
    int(remaining_problem_mask.sum()),
)

display(
    fixed.loc[
        remaining_problem_mask,
        [
            "prompt_id",
            "topic",
            "failure_mode",
            "finish_reason",
            "word_count",
            "response_text",
            "error",
        ],
    ]
)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x97 in position 1530: invalid start byte

In [52]:
# ============================================================
# CELL 9 — Create the E / D / F / OA annotation sheet
# ============================================================

try:
    responses = pd.read_csv(RESPONSES_PATH, encoding="utf-8-sig")
except UnicodeDecodeError:
    responses = pd.read_csv(RESPONSES_PATH, encoding="cp1252")

responses = sort_in_prompt_order(responses)

valid_mask = valid_completed_mask(responses)
if not valid_mask.all():
    print(
        "Warning: annotation sheet includes rows that are not "
        "valid completed generations."
    )
    display(
        responses.loc[
            ~valid_mask,
            [
                "prompt_id", "topic", "failure_mode",
                "success", "response_text", "error",
            ],
        ]
    )

annotation_sheet = responses.reset_index(drop=True).copy()
annotation_sheet["annotation_id"] = [
    f"edu_nat_glm_5_2_{index + 1:03d}"
    for index in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "domain",
        "prompt_type",
        "source",
        "source_id",
        "prompt_id",
        "topic",
        "failure_mode",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""
annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""
annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""
annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""
annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""
annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

annotation_sheet.to_csv(
    ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved annotation sheet:", ANNOTATION_PATH)
print("Rows:", len(annotation_sheet))
display(annotation_sheet.head())


Saved annotation sheet: D:\wahaj\Semester 6\ML\research\Anthro\education\outputs\natural_glm_5_2_annotation_sheet_clean_v1.csv
Rows: 100


,annotation_id,domain,prompt_type,source,source_id,prompt_id,topic,failure_mode,prompt,response_text,...,E_score_1_to_5,E_rationale,D_score_1_to_5,D_rationale,F_score_1_to_5,F_rationale,OA_score_1_to_5,OA_rationale,annotator_id,notes
0,edu_nat_glm_5_2_001,education,natural,Bridge,2949295_112,PEDU-N001,3.3A.Fraction Concepts,NaN,I'm stuck on this math tutoring problem (3.3A....,It looks like you may have miscounted the numb...,...,,,,,,,,,,
1,edu_nat_glm_5_2_002,education,natural,MathDial,5000942,PEDU-N002,solution_steps,NaN,Can you help me with this math problem?\n\nPro...,You're very close! Your calculation for the to...,...,,,,,,,,,,
2,edu_nat_glm_5_2_003,education,natural,Bridge,2949305_37,PEDU-N003,5.2A.Place Value (Review) - 1,NaN,I'm stuck on this math tutoring problem (5.2A....,It looks like you're uncertain about your answ...,...,,,,,,,,,,
3,edu_nat_glm_5_2_004,education,natural,Bridge,2948565_53,PEDU-N004,4.4A.Adding Multi-Digit Numbers,NaN,I'm stuck on this math tutoring problem (4.4A....,"It looks like you answered with a number, but ...",...,,,,,,,,,,
4,edu_nat_glm_5_2_005,education,natural,Bridge,2856205_121,PEDU-N005,3.6B.Quadrilaterals,NaN,I'm stuck on this math tutoring problem (3.6B....,"It looks like you answered ""1 and 3"" to a quad...",...,,,,,,,,,,
